# Build a point-in-time information set
A subject date says what a value describes. An availability timestamp says when a decision process could first use it. This lesson joins on availability.

## 1. Using pandas
`merge_asof` selects the latest eligible release at or before each decision time.

In [ ]:
import pandas as pd

releases = pd.DataFrame({
    'subject_period': pd.to_datetime(['2025-12-31', '2025-12-31', '2026-03-31']),
    'available': pd.to_datetime(['2026-02-15', '2026-03-20', '2026-05-10']),
    'version': ['initial', 'revised', 'initial'],
    'value': [10.0, 10.4, 11.1]
}).sort_values('available')
decisions = pd.DataFrame({'decision': pd.to_datetime(['2026-02-01', '2026-03-01', '2026-04-01', '2026-06-01'])})
point_in_time = pd.merge_asof(decisions.sort_values('decision'), releases,
                              left_on='decision', right_on='available', direction='backward')
point_in_time

## 2. Build the selection rule from scratch
Filter releases by availability and retain the latest eligible record.

In [ ]:
from datetime import date

release_rows = [(date(2026, 2, 15), 'initial', 10.0),
                (date(2026, 3, 20), 'revised', 10.4),
                (date(2026, 5, 10), 'initial', 11.1)]
decision_dates = [date(2026, 2, 1), date(2026, 3, 1), date(2026, 4, 1), date(2026, 6, 1)]

for decision in decision_dates:
    eligible = [row for row in release_rows if row[0] <= decision]
    latest = max(eligible, default=None, key=lambda row: row[0])
    print(decision, latest)

## What to notice
The 1 February decision sees no release. The 1 March decision sees the initial value, and the 1 April decision sees the revision. Joining on subject period alone would reveal both versions too early.